# Drug Review Insights & Summarization Tool: API Integration & Prompt Engineering

**Phase 3 — Alex**

This notebook:
- Loading  cleaned outputs (`cleaned_drug_reviews.csv`, `drug_review_profile.csv`)
- Builds structured data profiles for three query types: **overall dataset**, **per-drug**, and **per-condition**
- Sends each profile to the Gemini API and returns plain-language summaries
- Iterates on prompts to improve accuracy and relevance
- Saves all narrative outputs to `/data/final/ai_summaries.json`

Output file:
- `data/final/ai_summaries.json`

## Install & Import Packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import google.generativeai as genai

with open("/content/.gemini_key") as _f:
    api_key = _f.read().strip()
genai.configure(api_key=api_key)
print("Gemini client ready.")

Gemini client ready.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
import pandas as pd

cleaned_df = pd.read_csv("/content/cleaned_drug_reviews.csv")
profile_df = pd.read_csv("/content/drug_review_profile.csv")

print("Cleaned dataset shape:", cleaned_df.shape)
print("\nSummary profile:")
print(profile_df.to_string(index=False))

Cleaned dataset shape: (219206, 13)

Summary profile:
               metric         value
           total_rows 219206.000000
        total_columns     13.000000
       duplicate_rows     52.000000
         unique_drugs   4211.000000
    unique_conditions   2724.000000
       average_rating      6.989184
        median_rating      8.000000
average_review_length     85.335698


/tmp/ipykernel_20991/3320037781.py:3: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_df = pd.read_csv("/content/cleaned_drug_reviews.csv")


In [4]:
cleaned_df = pd.read_csv("/content/cleaned_drug_reviews.csv", low_memory=False)

## Build Structured Data Profiles for the AI

The AI cannot process 219K rows. Instead, we build **compact text summaries** that capture the key statistics and representative reviews for each query type:

1. **Overall dataset profile** — macro-level numbers + top drugs/conditions + sentiment breakdown
2. **Per-drug profile** — rating stats, sentiment split, and 3 sample reviews for a given drug
3. **Per-condition profile** — same structure, filtered by condition

These become the `{data_profile}` slot injected into each prompt.

In [5]:
def build_overall_profile(cleaned_df: pd.DataFrame, profile_df: pd.DataFrame) -> str:
    """
    Build a compact overall dataset profile string for the AI prompt.
    Pulls macro stats, rating distribution, top drugs/conditions,
    and sentiment breakdown from cleaned outputs.
    """
    # Summary statistics from profile_df
    stats = dict(zip(profile_df["metric"], profile_df["value"]))

    # Rating distribution
    rating_dist = (
        cleaned_df["rating"]
        .value_counts()
        .sort_index()
        .rename_axis("rating")
        .reset_index(name="count")
    )
    rating_str = ", ".join(
        f"rating {int(r)}: {int(c)} reviews"
        for r, c in zip(rating_dist["rating"], rating_dist["count"])
    )

    # Sentiment breakdown
    sentiment_counts = cleaned_df["sentiment"].value_counts()
    sentiment_str = ", ".join(
        f"{s}: {int(n)} ({100*n/len(cleaned_df):.1f}%)"
        for s, n in sentiment_counts.items()
    )

    # Top 10 drugs by review count
    top_drugs = (
        cleaned_df.groupby("drugName")
        .agg(review_count=("review", "count"), avg_rating=("rating", "mean"))
        .sort_values("review_count", ascending=False)
        .head(10)
        .reset_index()
    )
    drugs_str = "; ".join(
        f"{row.drugName} ({int(row.review_count)} reviews, avg rating {row.avg_rating:.1f})"
        for row in top_drugs.itertuples()
    )

    # Top 10 conditions by review count
    top_conditions = (
        cleaned_df.groupby("condition")
        .size()
        .sort_values(ascending=False)
        .head(10)
        .reset_index(name="count")
    )
    conditions_str = "; ".join(
        f"{row.condition} ({int(row.count)} reviews)"
        for row in top_conditions.itertuples()
    )

    profile = f"""OVERALL DATASET PROFILE
=======================
Total reviews: {int(stats.get('total_rows', 0)):,}
Unique drugs: {int(stats.get('unique_drugs', 0)):,}
Unique conditions: {int(stats.get('unique_conditions', 0)):,}
Average rating: {float(stats.get('average_rating', 0)):.2f} / 10
Median rating: {float(stats.get('median_rating', 0)):.1f} / 10
Average review length: {float(stats.get('average_review_length', 0)):.0f} words

Rating distribution: {rating_str}

Sentiment breakdown: {sentiment_str}

Top 10 drugs by review volume: {drugs_str}

Top 10 conditions by review volume: {conditions_str}
"""
    return profile.strip()


# Preview
overall_profile = build_overall_profile(cleaned_df, profile_df)
print(overall_profile)

OVERALL DATASET PROFILE
Total reviews: 219,206
Unique drugs: 4,211
Unique conditions: 2,724
Average rating: 6.99 / 10
Median rating: 8.0 / 10
Average review length: 85 words

Rating distribution: rating 1: 29338 reviews, rating 2: 9401 reviews, rating 3: 8913 reviews, rating 4: 6822 reviews, rating 5: 10949 reviews, rating 6: 8677 reviews, rating 7: 13018 reviews, rating 8: 25794 reviews, rating 9: 37321 reviews, rating 10: 68973 reviews

Sentiment breakdown: positive: 145106 (66.2%), negative: 54474 (24.9%), neutral: 19626 (9.0%)

Top 10 drugs by review volume: Levonorgestrel (4930 reviews, avg rating 7.4); Etonogestrel (4421 reviews, avg rating 5.8); Ethinyl estradiol / norethindrone (3753 reviews, avg rating 5.6); Nexplanon (2892 reviews, avg rating 5.6); Ethinyl estradiol / norgestimate (2790 reviews, avg rating 5.8); Ethinyl estradiol / levonorgestrel (2503 reviews, avg rating 5.8); Phentermine (2085 reviews, avg rating 8.8); Sertraline (1868 reviews, avg rating 7.5); Escitalopram

In [6]:
def build_drug_profile(cleaned_df: pd.DataFrame, drug_name: str, n_samples: int = 3) -> str:
    """
    Build a per-drug profile string.
    Includes rating stats, sentiment split, and up to n_samples representative reviews.
    """
    drug_df = cleaned_df[cleaned_df["drugName"].str.lower() == drug_name.lower()]

    if drug_df.empty:
        return f"No reviews found for drug: {drug_name}"

    review_count = len(drug_df)
    avg_rating = drug_df["rating"].mean()
    median_rating = drug_df["rating"].median()
    sentiment_counts = drug_df["sentiment"].value_counts()

    # Sample reviews: one positive, one neutral, one negative (if they exist)
    samples = []
    for s in ["positive", "neutral", "negative"]:
        subset = drug_df[drug_df["sentiment"] == s]
        if not subset.empty:
            row = subset.sample(1).iloc[0]
            # Truncate long reviews
            review_text = str(row["review"])[:400]
            samples.append(f'[{s.upper()}, rating {int(row["rating"])}] "{review_text}..."')

    samples_str = "\n".join(samples) if samples else "No sample reviews available."

    # Top conditions this drug is used for
    top_conditions = (
        drug_df.groupby("condition")
        .size()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )
    conditions_str = ", ".join(top_conditions)

    profile = f"""DRUG PROFILE: {drug_name.upper()}
{'='*(20+len(drug_name))}
Total reviews: {review_count:,}
Average rating: {avg_rating:.2f} / 10
Median rating: {median_rating:.1f} / 10
Positive reviews: {sentiment_counts.get('positive', 0)} ({100*sentiment_counts.get('positive',0)/review_count:.1f}%)
Neutral reviews:  {sentiment_counts.get('neutral', 0)} ({100*sentiment_counts.get('neutral',0)/review_count:.1f}%)
Negative reviews: {sentiment_counts.get('negative', 0)} ({100*sentiment_counts.get('negative',0)/review_count:.1f}%)

Top conditions treated: {conditions_str}

Sample patient reviews:
{samples_str}
"""
    return profile.strip()


# Preview with a common drug
drug_profile = build_drug_profile(cleaned_df, "Levothyroxine")
print(drug_profile)

DRUG PROFILE: LEVOTHYROXINE
Total reviews: 378
Average rating: 6.62 / 10
Median rating: 8.0 / 10
Positive reviews: 230 (60.8%)
Neutral reviews:  35 (9.3%)
Negative reviews: 113 (29.9%)

Top conditions treated: Underactive Thyroid, Hypothyroidism, After Thyroid Removal, Hashimoto's disease, TSH Suppression, Not Listed / Othe

Sample patient reviews:
[POSITIVE, rating 10] ""After trying 3 different medicines and a countless number of dosage combinations we found out that my body was not absorbing the tablets properly and the gel capsules allow my body to absorb the proper amounts. After having thyroid cancer at 15 and not being able to find anything that really worked I&#039;m glad we tried this. The only downside is the cost since it is almost triple what I have be..."
[NEUTRAL, rating 6] ""I first found out I had hypo thyroid 10/24 (OB checked bc I&#039;ve been trying to conceive) TSH Level 8.7. After taking 50mg for 1month my TSH level went to 3.0. Unfortunately after that I started g

In [7]:
def build_condition_profile(cleaned_df: pd.DataFrame, condition: str, n_drugs: int = 5, n_samples: int = 3) -> str:
    """
    Build a per-condition profile string.
    Includes top drugs for the condition, rating stats, sentiment split,
    and sample patient reviews.
    """
    cond_df = cleaned_df[cleaned_df["condition"].str.lower() == condition.lower()]

    if cond_df.empty:
        return f"No reviews found for condition: {condition}"

    review_count = len(cond_df)
    avg_rating = cond_df["rating"].mean()
    median_rating = cond_df["rating"].median()
    sentiment_counts = cond_df["sentiment"].value_counts()

    # Top drugs for this condition
    top_drugs = (
        cond_df.groupby("drugName")
        .agg(review_count=("review", "count"), avg_rating=("rating", "mean"))
        .sort_values("review_count", ascending=False)
        .head(n_drugs)
        .reset_index()
    )
    drugs_str = "; ".join(
        f"{row.drugName} ({int(row.review_count)} reviews, avg {row.avg_rating:.1f})"
        for row in top_drugs.itertuples()
    )

    # Sample reviews
    samples = []
    for s in ["positive", "neutral", "negative"]:
        subset = cond_df[cond_df["sentiment"] == s]
        if not subset.empty:
            row = subset.sample(1).iloc[0]
            review_text = str(row["review"])[:400]
            samples.append(f'[{s.upper()}, {row["drugName"]}, rating {int(row["rating"])}] "{review_text}..."')

    samples_str = "\n".join(samples) if samples else "No sample reviews available."

    profile = f"""CONDITION PROFILE: {condition.upper()}
{'='*(22+len(condition))}
Total reviews: {review_count:,}
Average rating: {avg_rating:.2f} / 10
Median rating: {median_rating:.1f} / 10
Positive reviews: {sentiment_counts.get('positive', 0)} ({100*sentiment_counts.get('positive',0)/review_count:.1f}%)
Neutral reviews:  {sentiment_counts.get('neutral', 0)} ({100*sentiment_counts.get('neutral',0)/review_count:.1f}%)
Negative reviews: {sentiment_counts.get('negative', 0)} ({100*sentiment_counts.get('negative',0)/review_count:.1f}%)

Top drugs for this condition: {drugs_str}

Sample patient reviews:
{samples_str}
"""
    return profile.strip()


# Preview
condition_profile = build_condition_profile(cleaned_df, "Depression")
print(condition_profile)

CONDITION PROFILE: DEPRESSION
Total reviews: 12,466
Average rating: 7.09 / 10
Median rating: 8.0 / 10
Positive reviews: 8555 (68.6%)
Neutral reviews:  1153 (9.2%)
Negative reviews: 2758 (22.1%)

Top drugs for this condition: Bupropion (747 reviews, avg 7.4); Sertraline (663 reviews, avg 7.1); Venlafaxine (574 reviews, avg 6.5); Desvenlafaxine (573 reviews, avg 7.2); Pristiq (554 reviews, avg 7.2)

Sample patient reviews:
[POSITIVE, Prozac, rating 10] ""I have had depression and anxiety for several years, and have been on many anti-depressants and none have been as effective as Prozac has been for me. I was previously on Celexa and although it did help my depression it also made me gain a lot of weight. Since being ok Prozac though my appetite has went down tremendously and I feel amazing. I have never been this happy even before depression hit ..."
[NEUTRAL, Desvenlafaxine, rating 5] ""Originally I would have given this drug 10/10. I had anxiety so bad that at one point I could not eve

## Prompt Engineering

We use three purpose-built prompts. The **system prompt** establishes the AI's role and output requirements. The **user prompt** injects the data profile via `{data_profile}`.

### Prompt Design Principles


In [8]:
# ── System prompt (shared across all query types) ────────────────────────────
SYSTEM_PROMPT = """You are a clinical data analyst assistant helping pharmacists and healthcare
researchers interpret patient drug review data. Your role is to produce clear, factual,
plain-language summaries of structured data profiles derived from patient-reported reviews.

Rules:
- Base every claim strictly on the numbers provided in the data profile. Do not infer or invent.
- Cite specific figures (averages, percentages, counts) to support each observation.
- Do not give medical advice or recommend specific treatments.
- Write in a professional but accessible tone appropriate for a pharmacist or researcher.
- Structure your response with clearly labeled sections as instructed.
- Keep each section concise (2–4 sentences).
"""

# ── User prompts (one per query type) ────────────────────────────────────────

OVERALL_PROMPT = """Below is a structured profile of a patient drug review dataset.

{data_profile}

Please produce a summary report with the following four sections:
1. Dataset Overview — size, scope, and key characteristics
2. Sentiment Landscape — overall positivity/negativity trends and what drives them
3. Notable Patterns — which drugs or conditions stand out and why
4. Limitations & Caveats — what a researcher should be cautious about when using this data
"""

DRUG_PROMPT = """Below is a structured profile of patient reviews for a specific medication.

{data_profile}

Please produce a medication summary report with the following four sections:
1. Patient Experience Overview — overall satisfaction level and rating patterns
2. Sentiment Breakdown — proportion of positive, neutral, and negative experiences and what the reviews reveal
3. Key Themes — recurring topics or concerns patients mention (infer only from the sample reviews provided)
4. Clinical Relevance — what a pharmacist should know when counseling patients on this medication
"""

CONDITION_PROMPT = """Below is a structured profile of patient reviews grouped by a specific medical condition.

{data_profile}

Please produce a condition summary report with the following four sections:
1. Condition Overview — patient volume, overall treatment satisfaction, and rating patterns
2. Drug Comparison — how top medications for this condition compare based on review counts and ratings
3. Patient Sentiment Themes — what positive and negative reviewers most commonly express (from sample reviews)
4. Research Implications — what gaps or patterns in this data are worth investigating further
"""

print("Prompts defined. System prompt length:", len(SYSTEM_PROMPT), "chars")
print("Overall prompt length:", len(OVERALL_PROMPT), "chars")
print("Drug prompt length:   ", len(DRUG_PROMPT), "chars")
print("Condition prompt length:", len(CONDITION_PROMPT), "chars")

Prompts defined. System prompt length: 716 chars
Overall prompt length: 446 chars
Drug prompt length:    567 chars
Condition prompt length: 583 chars


## Core API Call Function

In [9]:
def call_gemini(system_prompt: str, user_prompt_template: str, data_profile: str) -> str:
    """
    Send a data profile to Gemini and return the narrative summary.
    """
    user_message = user_prompt_template.format(data_profile=data_profile)

    full_prompt = f"{system_prompt}\n\n{user_message}"

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=full_prompt
    )

    return response.text.strip()

print("✓ Gemini API function ready.")

✓ Gemini API function ready.


## Generate Overall Dataset Summary

In [10]:
from google import genai

with open("/content/.gemini_key") as _f:
    api_key = _f.read().strip()
print("Key found:", api_key is not None)
print("Key preview:", api_key[:8] if api_key else "MISSING")

client = genai.Client(api_key=api_key)
print("Client:", client)

Key found: True
Key preview: AQ.Ab8RN


Client: <google.genai.client.Client object at 0x7bc27c4eddc0>


In [11]:
def call_gemini(system_prompt: str, user_prompt_template: str, data_profile: str) -> str:
    user_message = user_prompt_template.format(data_profile=data_profile)
    full_prompt = f"{system_prompt}\n\n{user_message}"
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=full_prompt
    )
    return response.text.strip()

print("✓ Gemini API function ready.")

✓ Gemini API function ready.


In [12]:
# Generate overall dataset summary
import time

def call_gemini(system_prompt: str, user_prompt_template: str, data_profile: str) -> str:
    user_message = user_prompt_template.format(data_profile=data_profile)
    full_prompt = f"{system_prompt}\n\n{user_message}"

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=full_prompt
            )
            return response.text.strip()
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)
                print(f"Rate limit hit. Waiting {wait} seconds...")
                time.sleep(wait)
            else:
                raise e

    raise Exception("Failed after 3 attempts.")

print("✓ Gemini 2.5 Flash ready.")

✓ Gemini 2.5 Flash ready.


In [13]:
print("Calling Gemini API...")

overall_summary = call_gemini(
    system_prompt=SYSTEM_PROMPT,
    user_prompt_template=OVERALL_PROMPT,
    data_profile=overall_profile
)

print("═" * 60)
print("OVERALL DATASET SUMMARY")
print("═" * 60)
print(overall_summary)

Calling Gemini API...


════════════════════════════════════════════════════════════
OVERALL DATASET SUMMARY
════════════════════════════════════════════════════════════
Here is a summary report of the patient drug review dataset:

### 1. Dataset Overview
This dataset comprises 219,206 patient drug reviews covering 4,211 unique drugs and 2,724 unique conditions. The average rating across all reviews is 6.99 out of 10, with a median rating of 8.0 out of 10. Each review has an average length of 85 words, indicating concise patient feedback. Notably, the highest volume of reviews is for a rating of 10 (68,973 reviews), followed by rating 1 (29,338 reviews), suggesting a bimodal distribution of satisfaction.

### 2. Sentiment Landscape
The overall sentiment in the dataset is predominantly positive, with 145,106 reviews (66.2%) categorized as such. Negative sentiment accounts for 54,474 reviews (24.9%), while neutral sentiment represents 19,626 reviews (9.0%). This high proportion of positive reviews is likely inf

## Generate Per-Drug Summaries (Top 5 Drugs by Review Volume)

In [14]:
# Top 5 drugs by review count
top_5_drugs = (
    cleaned_df.groupby("drugName")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

print("Generating summaries for:", top_5_drugs)

drug_summaries = {}

for drug in top_5_drugs:
    print(f"\n→ Calling API for: {drug}")
    profile = build_drug_profile(cleaned_df, drug)
    summary = call_gemini(
        system_prompt=SYSTEM_PROMPT,
        user_prompt_template=DRUG_PROMPT,
        data_profile=profile
    )
    drug_summaries[drug] = {
        "data_profile": profile,
        "summary": summary
    }
    print(f"  ✓ Done ({len(summary)} chars)")

print("\n✓ All drug summaries generated.")

Generating summaries for: ['Levonorgestrel', 'Etonogestrel', 'Ethinyl estradiol / norethindrone', 'Nexplanon', 'Ethinyl estradiol / norgestimate']

→ Calling API for: Levonorgestrel


  ✓ Done (2021 chars)

→ Calling API for: Etonogestrel


  ✓ Done (1748 chars)

→ Calling API for: Ethinyl estradiol / norethindrone


  ✓ Done (1640 chars)

→ Calling API for: Nexplanon


  ✓ Done (1559 chars)

→ Calling API for: Ethinyl estradiol / norgestimate


  ✓ Done (1814 chars)

✓ All drug summaries generated.


## Generate Per-Condition Summaries (Top 5 Conditions by Review Volume)

In [15]:
# Top 5 conditions by review count
top_5_conditions = (
    cleaned_df[cleaned_df["condition"] != "Unknown"]
    .groupby("condition")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

print("Generating summaries for:", top_5_conditions)

condition_summaries = {}

for condition in top_5_conditions:
    print(f"\n→ Calling API for: {condition}")
    profile = build_condition_profile(cleaned_df, condition)
    summary = call_gemini(
        system_prompt=SYSTEM_PROMPT,
        user_prompt_template=CONDITION_PROMPT,
        data_profile=profile
    )
    condition_summaries[condition] = {
        "data_profile": profile,
        "summary": summary
    }
    print(f"  ✓ Done ({len(summary)} chars)")

print("\n✓ All condition summaries generated.")

Generating summaries for: ['Birth Control', 'Depression', 'Pain', 'Anxiety', 'Acne']

→ Calling API for: Birth Control


Rate limit hit. Waiting 60 seconds...


  ✓ Done (2301 chars)

→ Calling API for: Depression


  ✓ Done (2250 chars)

→ Calling API for: Pain


  ✓ Done (2307 chars)

→ Calling API for: Anxiety


  ✓ Done (2469 chars)

→ Calling API for: Acne


  ✓ Done (2213 chars)

✓ All condition summaries generated.


In [16]:
# Inspect one example
example_drug = top_5_drugs[0]
print("═" * 60)
print(f"DRUG SUMMARY: {example_drug.upper()}")
print("═" * 60)
print(drug_summaries[example_drug]["summary"])

════════════════════════════════════════════════════════════
DRUG SUMMARY: LEVONORGESTREL
════════════════════════════════════════════════════════════
Here is a summary report based on the provided patient review data for Levonorgestrel:

### 1. Patient Experience Overview
Levonorgestrel has garnered a substantial number of patient reviews, totaling 4,930. The average rating for this medication is 7.38 out of 10. However, the median rating is notably higher at 9.0 out of 10, which suggests that while some patients may have very low ratings, a significant portion of reviewers reported generally positive experiences.

### 2. Sentiment Breakdown
The overall sentiment for Levonorgestrel is predominantly positive, with 3,435 reviews (69.7%) falling into this category. Negative experiences were reported by 1,007 patients (20.4%), indicating a noticeable segment of dissatisfaction. A smaller proportion of 488 reviews (9.9%) were classified as neutral.

### 3. Key Themes
Based on the provided 

## Save All Summaries to `ai_summaries.json`



In [17]:
import json
import os

output = {
    "overall": {
        "data_profile": overall_profile,
        "summary": overall_summary
    },
    "drugs": drug_summaries,
    "conditions": condition_summaries
}

os.makedirs("/content/data/final", exist_ok=True)

with open("/content/data/final/ai_summaries.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✓ Saved to /content/data/final/ai_summaries.json")
print(f"  Overall summary: ✓")
print(f"  Drug summaries: {len(drug_summaries)}")
print(f"  Condition summaries: {len(condition_summaries)}")

✓ Saved to /content/data/final/ai_summaries.json
  Overall summary: ✓
  Drug summaries: 5
  Condition summaries: 5
